# Complete Clinical RAG: Task 2 + Task 3

This notebook runs the optimized Task 2 retriever and the strictly grounded Task 3 generation layer as one pipeline. It loads the existing Chroma index and does not rebuild it automatically.

## Project structure

- `ingest.py`: PDF loading, chunking, embeddings, and index construction.
- `query.py`: existing Chroma index loading and scored retrieval.
- `grounding_prompt.py`: evidence-only prompt and context formatting.
- `response_schema.py`: formal grounded/refused response schema.
- `citation_validator.py`: independent citation checks.
- `llm_service.py`: Gemini API, Ollama, and deterministic simulation adapters.
- `pipeline.py`: retrieval gate, generation, JSON parsing, schema validation, and citation validation.
- `run_system.py`: command-line entry point.

In [ ]:
import csv
import json
import re

import config
from jsonschema import ValidationError, validate
from query import load_index, retrieve
from grounded_generation import (
    GROUNDING_SYSTEM_PROMPT,
    RESPONSE_SCHEMA,
    build_grounded_prompt,
    generate_grounded_response,
    validate_response,
)

## 1. Load and inspect the Task 2 retriever

In [ ]:
print('Embedding model:', config.EMBEDDING_MODEL)
print('Collection:', config.COLLECTION_NAME)
print('Chroma path:', config.CHROMA_DIR)
vectordb = load_index()
retrieval_question = 'What is the target blood pressure for a patient with cardiovascular disease?'
retrieved = retrieve(vectordb, retrieval_question, k=3)
assert retrieved, 'The existing Task 2 index returned no chunks'
for index, (document, score) in enumerate(retrieved, 1):
    metadata = document.metadata
    print(f'{index}. score={score:.4f} document={metadata.get("document_name")} page={metadata.get("page_number")} chunk={metadata.get("chunk_id")}')

In [ ]:
print('Top-k comparison using the existing retriever:')
for k in (1, 3, 5):
    results = retrieve(vectordb, retrieval_question, k=k)
    scores = [round(score, 4) for _, score in results]
    print(f'k={k}: scores={scores}')

In [ ]:
def expected_page(expected_source):
    match = re.search(r'Page (\d+)', expected_source)
    return int(match.group(1)) if match else None

with open(config.ROOT_DIR / 'eval' / 'Test_Set.csv', newline='', encoding='utf-8') as file_handle:
    evaluation_rows = list(csv.DictReader(file_handle))
scored_precisions = []
for row in evaluation_rows:
    page = expected_page(row['Expected Source (Document / Section / Page)'])
    if page is None:
        continue
    results = retrieve(vectordb, row['Question'], k=3)
    hits = sum(document.metadata.get('page_number') == page for document, _ in results)
    scored_precisions.append(hits / 3)
print(f'Average page-level Precision@3: {sum(scored_precisions) / len(scored_precisions):.3f}')

## 2. Grounding prompt, schema, and citations

In [ ]:
assert 'citation-bound clinical assistant' in GROUNDING_SYSTEM_PROMPT
assert 'ONLY the supplied retrieved context' in GROUNDING_SYSTEM_PROMPT
assert 'Return JSON only' in GROUNDING_SYSTEM_PROMPT
assert 'refused' in GROUNDING_SYSTEM_PROMPT
print(GROUNDING_SYSTEM_PROMPT)
print(json.dumps(RESPONSE_SCHEMA, indent=2)[:1600])
print(build_grounded_prompt(retrieval_question, retrieved)[:1800])

In [ ]:
valid_response = {'status': 'grounded', 'recommendation': 'Supported recommendation', 'evidence': ['Retrieved evidence'], 'citations': [{'document': retrieved[0][0].metadata['document_name'], 'page': retrieved[0][0].metadata['page_number']}], 'confidence': 'high'}
validate_response(valid_response, retrieved)
invalid_response = {'status': 'grounded', 'recommendation': 'Unsupported recommendation', 'evidence': [], 'citations': [], 'confidence': 'high'}
try:
    validate(invalid_response, RESPONSE_SCHEMA)
except ValidationError as error:
    print('Invalid high-confidence response rejected:', error.message)
else:
    raise AssertionError('Invalid response unexpectedly passed')
fake_citation = {'status': 'grounded', 'recommendation': 'Test', 'evidence': ['Test'], 'citations': [{'document': 'Fake Medical Guideline', 'page': 999}], 'confidence': 'high'}
try:
    validate_response(fake_citation, retrieved)
except ValueError as error:
    print('Fake citation rejected:', error)
else:
    raise AssertionError('Fake citation unexpectedly passed')

## 3. Grounded generation, Gemini, and refusal tests

In [ ]:
answer = generate_grounded_response(retrieval_question, vectordb, k=3)
validate_response(answer, retrieved)
assert answer['status'] == 'grounded' and answer['evidence'] and answer['citations']
print('Grounded generation passed:')
print(json.dumps(answer, indent=2))

refusal = generate_grounded_response('What is the recommended dietary salt intake for healthy adults?', vectordb, k=3, confidence_threshold=1.1)
validate(refusal, RESPONSE_SCHEMA)
assert refusal == {'status': 'refused', 'recommendation': None, 'evidence': [], 'citations': [], 'confidence': 'insufficient'}
print('Low-confidence/out-of-scope refusal passed:', refusal)

In [ ]:
for mode in ('invalid_schema', 'invalid_citation'):
    try:
        generate_grounded_response(retrieval_question, vectordb, k=3, simulation_mode=mode)
    except ValueError as error:
        print(f'Simulation {mode} rejected as expected:', str(error)[:140])
    else:
        raise AssertionError(f'Simulation {mode} unexpectedly passed')
print('Provider:', config.LLM_PROVIDER)
print('Gemini key configured:', bool(config.GEMINI_API_KEY))

## Final summary

Task 2 retrieval remains the source of evidence: the persisted Chroma index, embeddings, scores, document names, pages, and chunk IDs are reused. Task 3 adds the citation-bound prompt, JSON Schema, Gemini JSON generation, citation normalization and validation, confidence gating, deterministic simulation, and structured refusal. The command-line equivalent is `C:\Users\mahmo\anaconda3\python.exe run_system.py`. The original Task 2 and Task 3 notebooks remain available as historical references; use this notebook for the complete workflow.